In [ ]:
import lsdb

cone = lsdb.ConeSearch(ra=0, dec=0, radius_arcsec=150_000.0)

gaia = lsdb.open_catalog(
    "https://data.lsdb.io/hats/gaia_dr3", columns=["ra", "dec", "phot_g_mean_mag"]
)  # , search_filter=cone)

tess = lsdb.open_catalog("https://data.lsdb.io/hats/tess/tess_lightcurve")  # , columns = ['ra', 'dec',])

sdss = lsdb.open_catalog(
    "https://data.lsdb.io/hats/sdss_dr7_spectra"
)  # , columns = ['ra', 'dec', 'phot_g_mean_mag'])

In [ ]:
tess["lightcurve"].columns

In [ ]:
gaia

In [ ]:
gaia_query = gaia.query("phot_g_mean_mag < 20")

In [ ]:
gaia_x_tess = gaia.crossmatch(tess, suffix_method="overlapping_columns")

In [ ]:
gaia_mmu = gaia_x_tess.crossmatch(sdss)

In [ ]:
import pandas as pd
from lsdb.streams import CatalogStream, InfiniteStream

cat_stream = CatalogStream(gaia_mmu, partitions_per_chunk=2, shuffle=False)
# inf_stream = InfiniteStream(gaia_mmu, partitions_per_chunk=2, shuffle=False)

df = []

for i, chunk in enumerate(cat_stream):
    df.append(chunk)
    if i == 2:
        break

df = pd.concat(df, ignore_index=True)

In [ ]:
df

In [ ]:
df.iloc[0]["spectra_sdss_dr7_spectra"]

In [ ]:
df.iloc[0]["lightcurve_gaia_x_tess_lightcurve"]

In [ ]:
df.iloc[0]["ra_gaia_x_tess_lightcurve"]

In [ ]:
df.iloc[0]["dec_gaia_x_tess_lightcurve"]

In [ ]:
cat_iter = iter(cat_stream)
counter = 0

while True:
    try:
        chunk = next(cat_iter)
        print(f"Chunk {counter}: {len(chunk)} rows")
        counter += 1
        if counter == 2:
            break
    except StopIteration:
        break